In [51]:
import torch
import torch.nn as nn
import pandas as pd
from sklearn.model_selection import train_test_split
import ast

torch.manual_seed(42)

In [52]:
from rich import print as print

In [53]:
df = pd.read_csv('./data/preprocessed_yelp_data.csv')
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [54]:
len(train_df),len(test_df)

(800, 200)

In [55]:
train_df

,text,label
29,"['the', 'worst', 'was', 'the', 'salmon', 'sash...",0
535,"['an', 'excellent', 'new', 'restaurant', 'by',...",1
695,"['went', 'for', 'lunch', 'service', 'was', 'sl...",0
557,"['i', 'think', 'this', 'restaurant', 'suffers'...",0
836,"['just', 'had', 'lunch', 'here', 'and', 'had',...",1
...,...,...
106,"['the', 'food', 'was', 'delicious', 'our', 'ba...",1
270,"['the', 'veggitarian', 'platter', 'is', 'out',...",1
860,"['this', 'place', 'is', 'pretty', 'good', 'nic...",1
435,"['it', 'was', 'a', 'huge', 'awkward', '15lb', ...",0


In [56]:
test_df

,text,label
521,"['if', 'you', 'havent', 'gone', 'here', 'go', ...",1
737,"['try', 'them', 'in', 'the', 'airport', 'to', ...",1
740,"['the', 'restaurant', 'is', 'very', 'clean', '...",1
660,"['i', 'personally', 'love', 'the', 'hummus', '...",1
411,"['come', 'hungry', 'leave', 'happy', 'and', 's...",1
...,...,...
408,"['service', 'was', 'fantastic']",1
332,"['we', 'had', 'fantastic', 'service', 'and', '...",1
208,"['must', 'have', 'been', 'an', 'off', 'night',...",0
613,"['sorry', 'i', 'will', 'not', 'be', 'getting',...",0


In [57]:
def build_vocab(
        texts,
        max_size=10000,
        min_freq=2
):
    word_freq = {}
    for text in texts:
        for word in text:
            word_freq[word] = word_freq.get(word, 0) + 1
    word_freq = {w:f for w, f in word_freq.items() if f >= min_freq}
    
    vocab = {'<unk>': 0, '<pad>': 1}
    vocab.update({w: i+2 for i, (w, _) in enumerate(sorted(word_freq.items(), key=lambda x: -x[1]))})
    
    return {k: v for k, v in vocab.items() if v < max_size}

train_texts = train_df['text'].tolist()
train_texts = [ast.literal_eval(text) for text in train_texts]
vocab = build_vocab(train_texts)

In [58]:
len(vocab)

762

In [59]:
def numericalize(text, vocab):
    return [vocab.get(word, vocab['<unk>']) for word in text]

train_df['text'] = train_df['text'].apply(lambda x: numericalize(ast.literal_eval(x), vocab))
test_df['text'] = test_df['text'].apply(lambda x: numericalize(ast.literal_eval(x), vocab))

In [60]:
train_df['text'].iloc[0]

[2, 144, 5, 2, 303, 370]

In [61]:
from torch.utils.data import Dataset, DataLoader

class YelpDataset(Dataset):
    def __init__(self, dataframe, max_seq_length):
        self.texts = dataframe['text'].tolist()
        self.labels = dataframe['label'].tolist()
        self.max_seq_length = max_seq_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        if len(text) < self.max_seq_length:
            text = text + [1] * (self.max_seq_length - len(text))
        else:
            text = text[:self.max_seq_length]
        return torch.tensor(text, dtype=torch.long), torch.tensor(label, dtype=torch.float)

In [62]:
max_seq_length = 100
train_dataset = YelpDataset(train_df, max_seq_length)
test_dataset = YelpDataset(test_df, max_seq_length)
print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

Train dataset size: 800

Test dataset size: 200

In [63]:
train_dataset[0]

(tensor([  2, 144,   5,   2, 303, 370,   1,   1,   1,   1,   1,   1,   1,   1,
           1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
           1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
           1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
           1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
           1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
           1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
           1,   1]),
 tensor(0.))

In [64]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [65]:
class RNNModel(nn.Module):
    def __init__(self, input_size, embedding_dim, hidden_size, num_layers, output_size):
        super().__init__()
        self.embedding = nn.Embedding(input_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded)
        out = self.linear(hidden[-1])
        return out

In [66]:
input_size = len(vocab)
embedding_dim = 100
hidden_size = 256
num_layers = 2
output_size = 1

model = RNNModel(input_size, embedding_dim, hidden_size, num_layers, output_size)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params}")

Total parameters: 299689

In [67]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [68]:
def train_model(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    for texts, labels in loader:
        optimizer.zero_grad()
        predictions = model(texts).squeeze(1)
        loss = criterion(predictions, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate_model(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for texts, labels in loader:
            predictions = model(texts).squeeze(1)
            loss = criterion(predictions, labels)
            total_loss += loss.item()
            predicted = (torch.sigmoid(predictions) >= 0.5).float()
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), correct / total

In [69]:
num_epochs = 2
for epoch in range(num_epochs):
    train_loss = train_model(model, train_loader, criterion, optimizer)
    test_loss, test_acc = evaluate_model(model, test_loader, criterion)
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f} | Test Acc: {test_acc*100:.2f}%")

Epoch 1/2 | Train Loss: 0.7107 | Test Loss: 0.6918 | Test Acc: 52.00%

Epoch 2/2 | Train Loss: 0.7013 | Test Loss: 0.6919 | Test Acc: 52.00%